# TRAIN stage: fine-tuning, test evaluation and latency (Colab)

Fine-tunes pretrained YOLO variants with the shared augmentation policy (`configs/augmentation.yaml`),
evaluates them on the held-out **test** split and measures latency. Equivalent to `scripts/train_eval.py`,
but meant for a Colab GPU.

**Before running**
1. `Runtime > Change runtime type > T4 GPU`.
2. Put two zips in your Google Drive (names and folder are set in the config cell):
   - the repo code: `configs/`, `src/`, `pyproject.toml`
   - `data/processed/` generated by `scripts/process_data.py` (`images/`, `labels/`, `data.yaml`)
3. Run the cells in order. Training takes about 2 h per model on a T4.

Results are written to Drive (`OUT`), so they survive a Colab disconnection.

## 1. Config

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"   # Drive folder that holds the two zips
REPO_ZIP = "repozip.zip"        # contains configs/, src/, pyproject.toml
DATA_ZIP = "processed.zip"      # contains the contents of data/processed
OUT = "/content/drive/MyDrive/VPC2/runs"   # where weights, curves and CSVs are saved

# Recipes (arms of configs/augmentation.yaml) trained one after the other.
# 2 recipes x 2 models = 4 runs, about 7 h on a T4. Keep one recipe to train less.
POLICIES = ["proposed", "ultralytics_defaults"]
MODELS = ["yolov8s.pt", "yolov8n.pt"]
EPOCHS, BATCH, SEED = 50, 16, 42

## 2. Setup: Drive, dependencies, data and code

In [ ]:
assert "DRIVE_DIR" in globals(), "Run cell 1 (Config) first, then run this cell again"
import glob, os, sys
from google.colab import drive
drive.mount("/content/drive")

!pip install -q ultralytics==8.4.172

import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"

!rm -rf /content/repo /content/processed
!unzip -q -o "{DRIVE_DIR}/{REPO_ZIP}" -d /content/repo
!unzip -q -o "{DRIVE_DIR}/{DATA_ZIP}" -d /content/processed

# Folder that contains images/ and labels/ (the zip may add one extra level)
DATA_ROOT = os.path.dirname(glob.glob("/content/processed/**/images", recursive=True)[0])

# data.yaml with the paths of THIS machine (the one from process_data.py has local paths)
open(f"{DATA_ROOT}/data.yaml", "w").write(f"""path: {DATA_ROOT}
train: images/train
val: images/val
test: images/test
names:
  0: BIODEGRADABLE
  1: CARDBOARD
  2: GLASS
  3: METAL
  4: PAPER
  5: PLASTIC
""")

# Make the repo code (vpc2) importable
REPO_ROOT = os.path.dirname(glob.glob("/content/repo/**/configs", recursive=True)[0])
sys.path.append(os.path.join(REPO_ROOT, "src"))
os.chdir(REPO_ROOT)
os.makedirs(OUT, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT)
print("REPO_ROOT:", REPO_ROOT)
!ls {DATA_ROOT} {DATA_ROOT}/images

## 3. Train (one run per model; same data, seed, epochs and augmentation)

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from vpc2.data.augmentation import load_policy

for pol in POLICIES:
    policy = load_policy(pol)
    for w in MODELS:
        name = f"{Path(w).stem}_{pol}"          # e.g. yolov8s_proposed
        _ = YOLO(w).train(data=f"{DATA_ROOT}/data.yaml", epochs=EPOCHS, imgsz=640,
                          batch=BATCH, seed=SEED, workers=2, project=OUT, name=name,
                          exist_ok=True, **policy.train_kwargs())

## 4. Evaluate on test and measure latency

- Metrics are on the **test** split. The confusion matrices saved during training are on **validation**.
- Latency = end to end per image (read, preprocess, inference, postprocess), one image at a time,
  200 test images after a 20-image warm-up. Always report it together with the hardware (here: Colab T4).

In [ ]:
import time
import numpy as np
import pandas as pd

imgs = sorted(glob.glob(f"{DATA_ROOT}/images/test/*.*"))[:200]

for pol in POLICIES:
    metricas, latencias, por_clase = [], [], {}
    for w in MODELS:
        name = f"{Path(w).stem}_{pol}"
        model = YOLO(f"{OUT}/{name}/weights/best.pt")

        m = model.val(data=f"{DATA_ROOT}/data.yaml", split="test", imgsz=640, plots=False)
        p, r = m.box.mp, m.box.mr
        metricas.append({"modelo": name, "mAP50": round(m.box.map50, 4), "mAP50-95": round(m.box.map, 4),
                         "precision": round(p, 4), "recall": round(r, 4), "F1": round(2 * p * r / (p + r), 4)})
        por_clase[name] = dict(zip(m.names.values(), m.box.maps.round(4)))

        for f in imgs[:20]:                                   # warm-up
            model.predict(f, imgsz=640, verbose=False)
        t_ms = []
        for f in imgs:
            torch.cuda.synchronize(); t0 = time.perf_counter()
            model.predict(f, imgsz=640, verbose=False)
            torch.cuda.synchronize(); t_ms.append((time.perf_counter() - t0) * 1000)
        latencias.append({"modelo": name, "ms_medio": round(float(np.mean(t_ms)), 2),
                          "ms_mediana": round(float(np.median(t_ms)), 2),
                          "FPS": round(1000 / float(np.mean(t_ms)), 1)})

    tabla, lat, clases = pd.DataFrame(metricas), pd.DataFrame(latencias), pd.DataFrame(por_clase)
    tabla.to_csv(f"{OUT}/tabla_metricas_{pol}.csv", index=False)
    lat.to_csv(f"{OUT}/latencia_t4_{pol}.csv", index=False)
    clases.to_csv(f"{OUT}/map_por_clase_{pol}.csv")
    print(f"===== {pol} =====")
    display(tabla); display(lat); display(clases)

## Outputs (in `OUT`)
- `<model>_<policy>/weights/best.pt`: trained weights (use these for inference)
- `<model>_<policy>/args.yaml`, `results.csv`, `results.png`, `confusion_matrix*.png`
- `tabla_metricas_<policy>.csv`, `map_por_clase_<policy>.csv`, `latencia_t4_<policy>.csv`

Download the folder from Drive and place it under `reports/runs/` in the repo.